# Alpha-Mind Quick Start 9: Stock-Screening Backtest

> This example selects stocks with formulas and backtests the resulting strategy.

In [ ]:
import asyncio


## 1. Parameter Settings
----------------

* Rebalancing frequency: weekly
* Universe: CSI 800
* Start date: 2017-05-31
* End date: 2019-05-31
* Performance benchmark: CSI 300

In [ ]:
import datetime as dt
import numpy as np


## 2. Stock-Screening Formula
-----------------------

* On each rebalancing date, rank stocks by the `EMA5D` factor and select the three highest-ranked stocks.

In [ ]:
import pyarrow as pa


## 3. Calculation Process
----------------

* Fetch factor values and industry information.

In [ ]:
import pyarrow.compute as pc


* Select stocks with the factor and fetch their daily returns.

In [ ]:
rng = np.random.default_rng(42)
n_dates = 6


* Calculate each rebalancing result and its returns.

In [ ]:
n_assets = 12


## 4. Performance Presentation
-------------------

### 4.1 Daily Returns and Drawdown

In [ ]:
dates = [dt.datetime(2026, 1, 5) + dt.timedelta(days=index) for index in range(n_dates)]


In [ ]:
codes = np.array([f"{index:06d}" for index in range(1, n_assets + 1)])
n_rows = n_dates * n_assets


* Cumulative-return and drawdown curves

In [ ]:
panel = pa.table({
    "trade_date": [value for value in dates for _ in range(n_assets)],
    "code": np.tile(codes, n_dates),
    "factor": rng.normal(size=n_rows),
    "value": rng.normal(size=n_rows),
    "dx": rng.normal(0.0, 0.01, size=n_rows),
    "industry": np.tile(np.array(["Bank", "Auto", "Health"]), n_dates * n_assets // 3),
    "benchmark_weight": np.full(n_rows, 1.0 / n_assets),
    "BETA": np.tile(np.linspace(0.8, 1.2, n_assets) ** 2, n_dates),
    "SIZE": np.tile(np.linspace(-1.0, 1.0, n_assets), n_dates),
})


* Maximum drawdown and annualized return

In [ ]:
from alphamind.data.transformer import Transformer


### 4.2 Excess Return and Alpha

In [ ]:
from alphamind.execution.naiveexecutor import NaiveExecutor
from alphamind.formula import CSRank


In [ ]:
from alphamind.portfolio.rankbuilder import rank_build


In [ ]:
scores = await asyncio.to_thread(
    Transformer({"score": CSRank("factor")}).transform, "code", panel
)


### 4.3 Monthly Returns

In [ ]:
ranked_panel = panel.append_column("score", scores["score"])
executor = NaiveExecutor()


### 4.4 Industry Distribution

In [ ]:
results = []


### 4.5 Turnover

* Monthly-turnover histogram

In [ ]:
for trade_date in dates:
    section = ranked_panel.filter(pc.equal(ranked_panel["trade_date"], trade_date))
    weights = rank_build(
        section["score"].to_numpy(zero_copy_only=False), use_rank=3
    ).flatten().astype(float) / 3.0
    target = pa.table({"code": section["code"], "weight": weights})
    turn_over, executed = executor.execute(target)
    executor.set_current(executed)
    result = float(weights @ section["dx"].to_numpy(zero_copy_only=False))
    results.append(pa.table({
        "trade_date": [trade_date], "return": [result], "turn_over": [turn_over]
    }))


In [ ]:
backtest = pa.concat_tables(results)
assert backtest.num_rows == n_dates


In [ ]:
backtest
